In [23]:
#https://judge.nitro-ai.org/competitions/roai-2025/nationala-ix-x-2026/1/view

In [15]:
import numpy as np
import pandas as pd
import fasttext
from gensim.models import KeyedVectors
from tqdm import tqdm
from scipy.linalg import orthogonal_procrustes
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
sample = pd.read_csv("/kaggle/input/datasets/umirzoq001/future-text-understanding/sample.csv")
test = pd.read_csv("/kaggle/input/datasets/umirzoq001/future-text-understanding/test.csv")
train = pd.read_csv("/kaggle/input/datasets/umirzoq001/future-text-understanding/train.csv")

In [10]:
eng_model = KeyedVectors.load_word2vec_format(
    "/kaggle/input/datasets/umirzoq001/future-text-understanding/starter/wiki.en.small.vec", 
    binary=False
)
print("English model loaded successfully!")

print("Loading Romanian model...")
ro_model = KeyedVectors.load_word2vec_format(
    "/kaggle/input/datasets/umirzoq001/future-text-understanding/starter/wiki.ro.small.vec", 
    binary=False
)
print("Romanian model loaded successfully!")


English model loaded successfully!
Loading Romanian model...
Romanian model loaded successfully!


In [4]:
train

,english,romanian
0,"If you add three and four, you get seven.","Dacă aduni trei cu patru, obții șapte."
1,"At first, I didn't like it, but it gradually b...","La început nu mi-a plăcut, dar treptat a deven..."
2,He made me love jazz.,El m-a făcut să îndrăgesc jazz-ul.
3,What time will you leave?,La ce oră vei pleca?
4,She went with him to Boston.,Ea a plecat cu el la Boston.
...,...,...
15177,How many boys are there in your class?,Câți băieți sunt în clasa ta?
15178,Did you enjoy going to exhibits in Romania?,Îți plăcea să mergi la expoziții în România?
15179,Darn!,Naiba!
15180,Existence is a meaningless concept.,Existența este un concept fără sens.


In [5]:
test

,datapointID,Word,Similar
0,1,beverage,pasionat
1,2,analyze,adopta
2,3,film,audio
3,4,dome,vultur
4,5,enlarge,atenție
...,...,...,...
1110,1111,classify,elimina
1111,1112,anywhere,seară
1112,1113,fill,emisie
1113,1114,fort,chimie


In [6]:
sample

,subtaskID,datapointID,answer
0,1,0,18
1,2,1,paisprezece
2,2,2,trage
3,2,3,graniță
4,2,4,stricat
...,...,...,...
1111,2,1111,glonț
1112,2,1112,armură
1113,2,1113,baschet
1114,2,1114,direcție


In [13]:
english_words = set(test['Word'].dropna().str.lower())
romanian_words = set(test['Similar'].dropna().str.lower())
subtask1_ans = len(english_words.intersection(romanian_words))

In [16]:
def get_sentence_vec(text, model):
    words = str(text).lower().split()
    vecs = [model[w] for w in words if w in model]
    if vecs:
        return np.mean(vecs, axis=0)
    return np.zeros(model.vector_size)

col_en = train.columns[0]
col_ro = train.columns[1]

X_train = np.array([get_sentence_vec(text, eng_model) for text in train[col_en]])
Y_train = np.array([get_sentence_vec(text, ro_model) for text in train[col_ro]])

valid_idx = (np.linalg.norm(X_train, axis=1) > 0) & (np.linalg.norm(Y_train, axis=1) > 0)
X_train = X_train[valid_idx]
Y_train = Y_train[valid_idx]

W, _ = orthogonal_procrustes(X_train, Y_train)

In [20]:
from scipy.optimize import linear_sum_assignment

test_en_words = test['Word'].tolist()
test_ro_words = test['Similar'].dropna().tolist()

X_test = np.array([eng_model[w] if w in eng_model else np.zeros(eng_model.vector_size) for w in test_en_words])
Y_cand = np.array([ro_model[w] if w in ro_model else np.zeros(ro_model.vector_size) for w in test_ro_words])

X_test_mapped = X_test.dot(W)

sim_matrix = cosine_similarity(X_test_mapped, Y_cand)

row_ind, col_ind = linear_sum_assignment(-sim_matrix)

predictions = [None] * len(test_en_words)
for r, c in zip(row_ind, col_ind):
    predictions[r] = test_ro_words[c]

In [21]:
subtask1_df = pd.DataFrame({
    'subtaskID': [1],
    'datapointID': [0],
    'answer': [subtask1_ans]
})

dp_id = test['id'] if 'id' in test.columns else test.index + 1
subtask2_df = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': dp_id,
    'answer': predictions
})

final_submission = pd.concat([subtask1_df, subtask2_df], ignore_index=True)
final_submission.to_csv('submission.csv', index=False)

In [22]:
final_submission

,subtaskID,datapointID,answer
0,1,0,124
1,2,1,băutură
2,2,2,analiza
3,2,3,film
4,2,4,cupolă
...,...,...,...
1111,2,1111,clasifica
1112,2,1112,oriunde
1113,2,1113,umple
1114,2,1114,fort
